# Query Decomposition

A question like "Compare the climate policies of Germany and Japan and explain how each affects their automotive industry" is too compound for a single vector search. The query touches climate policy, Germany, Japan, and the automotive sector across at least four distinct information needs. Embedding the full question and searching once produces a result set that partially satisfies each need but fully satisfies none. Query decomposition solves this by using the LLM to break the compound question into **independent, focused sub-queries**, retrieving for each separately, and then synthesizing a single answer from all the retrieved context.

**What you'll build:** A retrieval pipeline that decomposes complex questions into sub-queries, retrieves documents for each in parallel, deduplicates the combined result set, and synthesizes a coherent answer.

**Time:** ~20 min | **Difficulty:** Intermediate

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- How `QueryDecomposer` generates and structures sub-queries
- Why parallel retrieval per sub-query outperforms a single compound-query search
- How to configure the synthesis step to produce coherent multi-part answers
- How to inspect and override the generated sub-queries

In [ ]:
!pip install synapsekit -q

## Environment Setup

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.retrievers import DecompositionRetriever

llm = OpenAILLM(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Load a document corpus covering multiple topics

In [ ]:
docs = [
    # Climate policy — Germany
    "Germany's Energiewende policy targets 80% renewable electricity by 2030 and carbon neutrality by 2045.",
    "Germany's carbon pricing system covers industry and transport under the national emissions trading scheme.",
    "Germany has invested over \u20ac50 billion in offshore wind capacity since 2010.",

    # Climate policy — Japan
    "Japan's Green Growth Strategy targets carbon neutrality by 2050, focusing on hydrogen and ammonia power.",
    "Japan introduced a carbon tax in 2012 at a rate of \u00a5289 per tonne of CO2.",
    "Japan's feed-in tariff scheme accelerated solar deployment, adding 70GW of capacity by 2023.",

    # Automotive industry — Germany
    "Volkswagen, BMW, and Mercedes-Benz are investing over \u20ac300 billion in EV transition through 2030.",
    "Germany's automotive sector employs 800,000 workers directly and faces significant transition risk.",
    "The EU's 2035 combustion engine ban directly affects German OEMs, accelerating their EV roadmaps.",

    # Automotive industry — Japan
    "Toyota leads Japan's automotive sector with the world's best-selling hybrid lineup.",
    "Japan's automotive industry accounts for 10% of GDP and 20% of total exports.",
    "Honda and Nissan announced a merger in 2024 to pool EV development resources.",
]

await vectorstore.aadd(docs)
print(f"Added {len(docs)} documents to the vector store.")

## Step 3: Configure the DecompositionRetriever

`max_subqueries` caps the number of sub-queries generated. For most compound questions, 3-5 sub-queries cover the independent information needs without over-fragmenting.

In [ ]:
retriever = DecompositionRetriever(
    vectorstore=vectorstore,
    llm=llm,
    max_subqueries=5,      # upper bound on decomposition depth
    top_k_per_query=3,     # documents retrieved per sub-query before deduplication
    deduplicate=True,      # remove duplicate documents across sub-query result sets
)
print("DecompositionRetriever configured.")

## Step 4: Inspect the decomposition

Before running a full pipeline, inspect the sub-queries the LLM generates to verify they are independent and cover the question's information needs.

In [ ]:
async def inspect_decomposition():
    question = "Compare the climate policies of Germany and Japan and explain how each affects their automotive industry."
    subqueries = await retriever.adecompose(question)
    print(f"Decomposed '{question[:50]}...' into {len(subqueries)} sub-queries:\n")
    for i, sq in enumerate(subqueries, 1):
        print(f"  {i}. {sq}")

asyncio.run(inspect_decomposition())

## Step 5: Retrieve across sub-queries

In [ ]:
async def decomposed_retrieve():
    question = "Compare the climate policies of Germany and Japan and explain how each affects their automotive industry."
    results = await retriever.aretrieve(question)
    print(f"Total documents after deduplication: {len(results)}\n")
    for doc, score in results:
        print(f"[{score:.3f}] {doc[:80]}")

asyncio.run(decomposed_retrieve())

## Step 6: Wire into a RAG pipeline and ask the full question

In [ ]:
rag = RAG(llm=llm, retriever=retriever)
print("RAG pipeline assembled.")

## Step 7: Stream a structured answer

In [ ]:
async def ask(question: str):
    print(f"Q: {question}\n")
    async for chunk in rag.astream(question):
        print(chunk, end="", flush=True)
    print()

asyncio.run(ask(
    "Compare the climate policies of Germany and Japan and explain how each affects their automotive industry."
))

## Step 8: Handle a sequential decomposition (step-by-step reasoning)

Some questions have dependencies between sub-queries — the answer to sub-query 2 depends on the answer to sub-query 1. Use `mode="sequential"` to answer sub-queries in order, passing each answer as context for the next.

In [ ]:
retriever_seq = DecompositionRetriever(
    vectorstore=vectorstore,
    llm=llm,
    max_subqueries=4,
    top_k_per_query=3,
    mode="sequential",  # answer each sub-query in order; prior answers inform later retrievals
)

rag_seq = RAG(llm=llm, retriever=retriever_seq)
print("Sequential decomposition retriever configured.")

## Complete Working Example

A single self-contained cell that runs the entire Query Decomposition pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.retrievers import DecompositionRetriever

DOCS = [
    "Germany's Energiewende policy targets 80% renewable electricity by 2030 and carbon neutrality by 2045.",
    "Germany's carbon pricing system covers industry and transport under the national emissions trading scheme.",
    "Germany has invested over \u20ac50 billion in offshore wind capacity since 2010.",
    "Japan's Green Growth Strategy targets carbon neutrality by 2050, focusing on hydrogen and ammonia power.",
    "Japan introduced a carbon tax in 2012 at a rate of \u00a5289 per tonne of CO2.",
    "Japan's feed-in tariff scheme accelerated solar deployment, adding 70GW of capacity by 2023.",
    "Volkswagen, BMW, and Mercedes-Benz are investing over \u20ac300 billion in EV transition through 2030.",
    "Germany's automotive sector employs 800,000 workers directly and faces significant transition risk.",
    "The EU's 2035 combustion engine ban directly affects German OEMs, accelerating their EV roadmaps.",
    "Toyota leads Japan's automotive sector with the world's best-selling hybrid lineup.",
    "Japan's automotive industry accounts for 10% of GDP and 20% of total exports.",
    "Honda and Nissan announced a merger in 2024 to pool EV development resources.",
]


async def main():
    llm = OpenAILLM(model="gpt-4o-mini")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    await vectorstore.aadd(DOCS)

    retriever = DecompositionRetriever(
        vectorstore=vectorstore,
        llm=llm,
        max_subqueries=5,
        top_k_per_query=3,
        deduplicate=True,
    )

    rag = RAG(llm=llm, retriever=retriever)

    question = (
        "Compare the climate policies of Germany and Japan and explain "
        "how each affects their automotive industry."
    )

    # Show what sub-queries were generated
    subqueries = await retriever.adecompose(question)
    print("Sub-queries generated:")
    for i, sq in enumerate(subqueries, 1):
        print(f"  {i}. {sq}")
    print()

    # Run the full pipeline
    print(f"Q: {question}\n")
    result = await rag.aquery(question)
    print(f"A: {result.answer}")


asyncio.run(main())

## Next Steps

- [RAG Fusion](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/rag-fusion) — apply multi-query fusion within each sub-query for even better per-sub-query retrieval
- [GraphRAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/graph-rag) — for multi-hop questions where sub-queries have entity dependencies
- [Cross-Encoder Reranking](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/cross-encoder-reranking) — rerank the deduplicated combined result set before synthesis